# Анализ лояльности пользователей Яндекс Афиши


## Цели и задачи проекта:

С помощью датасета с информацией о клиентах и их активности, надо проанализировать и выявить портрет пользователя, который с большей вероятностью будет возвращаться на платформу и делать повторные заказы. Это поможет оптимизировать рекламные бюжеты и влиять на продолжительность удержания клиента

## Описание данных:
- user_id — уникальный идентификатор пользователя, совершившего заказ;
- device_type_canonical — тип устройства, с которого был оформлен заказ (mobile — мобильные устройства, desktop — стационарные);
- order_id — уникальный идентификатор заказа;
- order_dt — дата создания заказа (используйте данные created_dt_msk);
- order_ts — дата и время создания заказа (используйте данные created_ts_msk);
- currency_code — валюта оплаты;
- revenue — выручка от заказа;
- tickets_count — количество купленных билетов;
- days_since_prev — количество дней от предыдущей покупки пользователя, для пользователей с одной покупкой — значение пропущено;
- event_id — уникальный идентификатор мероприятия;
- service_name — название билетного оператора;
event_type_main — основной тип мероприятия (театральная постановка, концерт и так далее);
- region_name — название региона, в котором прошло мероприятие;
- city_name — название города, в котором прошло мероприятие.


### 0.Установка необходимых библиотек

In [1]:
#Установка необходимых библиотек
!pip install sqlalchemy

In [2]:
!pip install psycopg2

In [3]:
#Импорт необходимых библиотек для работы
#Для анализа данных
import pandas as pd

#Для визуализации
import matplotlib.pyplot as plt
import seaborn as sns

#Для подключения к БД
import os 
from sqlalchemy import create_engine
from dotenv import load_dotenv

In [4]:
#Получаю доступ к БД, используя данные из файла .env
db_config = {
    'user': os.getenv('DB_USER'),
    'pwd': os.getenv('DB_PASSWORD'),
    'host': os.getenv('DB_HOST'),
    'port': os.getenv('DB_PORT'),
    'db': os.getenv('DB_NAME')
}

In [5]:
#Создаю данные для подключения
connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

In [6]:
#Проверяю результат 
db_config

{'user': 'praktikum_student',
 'pwd': 'Sdf4$2;d-d30pp',
 'host': 'rc1b-wcoijxj3yxfsf3fs.mdb.yandexcloud.net',
 'port': '6432',
 'db': 'data-analyst-afisha'}

In [7]:
engine = create_engine(connection_string)

In [8]:
#Импортируем SQL-запрос 
query = '''
-- Настройка параметра synchronize_seqscans важна для проверки
WITH set_config_precode AS (
  SELECT set_config('synchronize_seqscans', 'off', true)
)
-- Напишите ваш запрос ниже
SELECT p.user_id,
      p.device_type_canonical,
      p.order_id,
      p.created_dt_msk AS order_dt,
      p.created_ts_msk AS order_ts,
      p.currency_code,
      p.revenue,
      p.tickets_count,
      EXTRACT (DAY FROM (p.created_dt_msk - LAG(p.created_dt_msk) OVER (PARTITION BY p.user_id ORDER BY p.created_dt_msk)))::INTEGER AS days_since_prev,
      p.event_id,
      e.event_name_code AS event_name,
      e.event_type_main,
      p.service_name,
      r.region_name,
      c.city_name
FROM afisha.purchases p
JOIN afisha.events e USING(event_id)
JOIN afisha.city c USING(city_id)
JOIN afisha.regions r USING(region_id)
WHERE p.device_type_canonical IN ('mobile', 'desktop') AND e.event_type_main NOT IN ('фильм')
ORDER BY p.user_id;
''' 

In [9]:
#Присваиваю результат SQL-запроса в переменную
df = pd.read_sql_query(query, con=engine)


In [10]:
#Проверяю результат выгрузки датасета
df.head()

,user_id,device_type_canonical,order_id,order_dt,order_ts,currency_code,revenue,tickets_count,days_since_prev,event_id,event_name,event_type_main,service_name,region_name,city_name
0,0002849b70a3ce2,mobile,4359165,2024-08-20,2024-08-20 16:08:03,rub,1521.94,4,NaN,169230,f0f7b271-04eb-4af6-bcb8-8f05cf46d6ad,театр,Край билетов,Каменевский регион,Глиногорск
1,0005ca5e93f2cf4,mobile,7965605,2024-07-23,2024-07-23 18:36:24,rub,289.45,2,NaN,237325,40efeb04-81b7-4135-b41f-708ff00cc64c,выставки,Мой билет,Каменевский регион,Глиногорск
2,0005ca5e93f2cf4,mobile,7292370,2024-10-06,2024-10-06 13:56:02,rub,1258.57,4,75.0,578454,01f3fb7b-ed07-4f94-b1d3-9a2e1ee5a8ca,другое,За билетом!,Каменевский регион,Глиногорск
3,000898990054619,mobile,1139875,2024-07-13,2024-07-13 19:40:48,rub,8.49,2,NaN,387271,2f638715-8844-466c-b43f-378a627c419f,другое,Лови билет!,Североярская область,Озёрск
4,000898990054619,mobile,972400,2024-10-04,2024-10-04 22:33:15,rub,1390.41,3,83.0,509453,10d805d3-9809-4d8a-834e-225b7d03f95d,стендап,Билеты без проблем,Озернинский край,Родниковецк


**Приведение названий клолонок к типу SnakeCase циклом**
for col in df.columns:
    str_bad = col
    if str_bad[0].isupper():
        str_bad = str_bad[0].lower() + str_bad[1:]
    for index_s, item_s in enumerate(str_bad):
        if item_s.isupper():
            str_bad = str_bad[:index_s] + '_' + str_bad[index_s].lower() + str_bad[index_s + 1:]
    df = df.rename(columns={col: str_bad})
return df

### 1. Знакомство с данными

In [11]:
#Узнаем размер датасета
df.shape

(290611, 15)

In [12]:
#Вывожу названия всех колонок списком
df.columns.to_list()

['user_id',
 'device_type_canonical',
 'order_id',
 'order_dt',
 'order_ts',
 'currency_code',
 'revenue',
 'tickets_count',
 'days_since_prev',
 'event_id',
 'event_name',
 'event_type_main',
 'service_name',
 'region_name',
 'city_name']

In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 290611 entries, 0 to 290610
Data columns (total 15 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   user_id                290611 non-null  object        
 1   device_type_canonical  290611 non-null  object        
 2   order_id               290611 non-null  int64         
 3   order_dt               290611 non-null  datetime64[ns]
 4   order_ts               290611 non-null  datetime64[ns]
 5   currency_code          290611 non-null  object        
 6   revenue                290611 non-null  float64       
 7   tickets_count          290611 non-null  int64         
 8   days_since_prev        268678 non-null  float64       
 9   event_id               290611 non-null  int64         
 10  event_name             290611 non-null  object        
 11  event_type_main        290611 non-null  object        
 12  service_name           290611 non-null  obje

#### Промежуточные выводы: 
- датасет польностью выгрузился без потери данных
- размер датасета чуть более 290к строк и 15колонок
- вижу пропущенные значения в колонке "days_since_prev"
- названия колонок приведены к типу Snake_Case


## Шаг2. Предобработка данных

#### Задача 2.1. 
Данные о выручке сервиса представлены в российских рублях и казахстанских тенге. Приведите выручку к единой валюте — российскому рублю.

In [14]:
#Импортируем файл с информацией о продажах в тенге
df_tenge = pd.read_csv('final_tickets_tenge_df.csv')

In [15]:
#Проверяю выгрузку файла
df_tenge.head()


,data,nominal,curs,cdx
0,2024-01-10,100,19.9391,kzt
1,2024-01-11,100,19.7255,kzt
2,2024-01-12,100,19.5839,kzt
3,2024-01-13,100,19.4501,kzt
4,2024-01-14,100,19.4501,kzt


In [16]:
#Самый простой способ вывести новые значения для ячеек с информацией о выручке в тенге, это соединить 2 таблицы и рассчитать данные по новому полю, не трогая значения в рублях
#Смотрю на типы данных в новом датафрейме
df_tenge.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 357 entries, 0 to 356
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   data     357 non-null    object 
 1   nominal  357 non-null    int64  
 2   curs     357 non-null    float64
 3   cdx      357 non-null    object 
dtypes: float64(1), int64(1), object(2)
memory usage: 11.3+ KB


In [17]:
#Будем соединять таблицы исходя из даты
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 290611 entries, 0 to 290610
Data columns (total 15 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   user_id                290611 non-null  object        
 1   device_type_canonical  290611 non-null  object        
 2   order_id               290611 non-null  int64         
 3   order_dt               290611 non-null  datetime64[ns]
 4   order_ts               290611 non-null  datetime64[ns]
 5   currency_code          290611 non-null  object        
 6   revenue                290611 non-null  float64       
 7   tickets_count          290611 non-null  int64         
 8   days_since_prev        268678 non-null  float64       
 9   event_id               290611 non-null  int64         
 10  event_name             290611 non-null  object        
 11  event_type_main        290611 non-null  object        
 12  service_name           290611 non-null  obje

In [18]:
#Меняем тип данных для df_tenge
df_tenge['data'] = pd.to_datetime(df_tenge['data'])

In [19]:
#Проверяю результат
df_tenge.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 357 entries, 0 to 356
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   data     357 non-null    datetime64[ns]
 1   nominal  357 non-null    int64         
 2   curs     357 non-null    float64       
 3   cdx      357 non-null    object        
dtypes: datetime64[ns](1), float64(1), int64(1), object(1)
memory usage: 11.3+ KB


In [25]:
#Теперь можно объединять таблицы
#Объединяем 2 таблицы так, чтоюы в результатирующей добавилась только колонка "curs"
df = df.merge(df_tenge[['data', 'curs']], how='left', left_on='order_dt', right_on='data').drop('data', axis=1)

In [27]:
#Теперь можно создавать новое поле с выручкой, где будет перерасчет сумм из тенге
df['revenue_rub'] = df['revenue'].where(df['currency_code']=='rub', (df['revenue'] * df['curs']) / 100)

In [29]:
#Проверяю результат 
df[df['currency_code'] == 'kzt'].head()


,user_id,device_type_canonical,order_id,order_dt,order_ts,currency_code,revenue,tickets_count,days_since_prev,event_id,event_name,event_type_main,service_name,region_name,city_name,curs,revenue_rub
70,0033403583a55ed,mobile,7850214,2024-09-17,2024-09-17 16:52:06,kzt,518.10,4,20.0,559919,f5ec2467-ee48-42f3-89a0-a2d90d1d6fce,другое,Билеты без проблем,Верхоречная область,Серебрянка,19.0125,98.503762
89,0040e12d70fae81,desktop,8634925,2024-09-02,2024-09-02 19:49:14,kzt,347.18,3,39.0,559476,61ec7c0e-ea0c-4a3e-a63a-09cbe1bc6177,другое,Билеты без проблем,Верхоречная область,Серебрянка,18.9330,65.731589
96,0054b38b2653e6e,mobile,7625522,2024-09-09,2024-09-09 18:06:04,kzt,328.77,4,NaN,559582,524f434e-0381-4f47-9688-7c4e41961b65,другое,Билеты без проблем,Верхоречная область,Серебрянка,18.5991,61.148261
277,00f4f5244501ce6,mobile,720419,2024-06-11,2024-06-11 20:15:57,kzt,22021.55,5,NaN,536826,76af4f88-c554-4e9e-9df4-1cc8aeec8959,концерты,Мой билет,Верхоречная область,Серебрянка,19.8928,4380.702898
460,01370a0e9c1a7d5,desktop,5227657,2024-06-04,2024-06-04 13:28:17,kzt,7397.66,4,1.0,533803,0686ac7b-6f3e-4177-bb73-16b54acd1223,концерты,Мой билет,Шанырский регион,Широковка,19.9833,1478.296591


In [30]:
#Проверяю размер результатирующего датасета
df.shape

(290611, 17)

**Пропущенные значения**

In [31]:
#Посмотрим где есть пропуска и сколько
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 290611 entries, 0 to 290610
Data columns (total 17 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   user_id                290611 non-null  object        
 1   device_type_canonical  290611 non-null  object        
 2   order_id               290611 non-null  int64         
 3   order_dt               290611 non-null  datetime64[ns]
 4   order_ts               290611 non-null  datetime64[ns]
 5   currency_code          290611 non-null  object        
 6   revenue                290611 non-null  float64       
 7   tickets_count          290611 non-null  int64         
 8   days_since_prev        268678 non-null  float64       
 9   event_id               290611 non-null  int64         
 10  event_name             290611 non-null  object        
 11  event_type_main        290611 non-null  object        
 12  service_name           290611 non-null  obje

In [34]:
#Считаем пропуски в абсолютном значении
df.isna().sum()

user_id                      0
device_type_canonical        0
order_id                     0
order_dt                     0
order_ts                     0
currency_code                0
revenue                      0
tickets_count                0
days_since_prev          21933
event_id                     0
event_name                   0
event_type_main              0
service_name                 0
region_name                  0
city_name                    0
curs                         0
revenue_rub                  0
dtype: int64

In [36]:
#Считаем пропуски в относительном значении
(df.isna().sum() / df.shape[0]) * 100

user_id                  0.000000
device_type_canonical    0.000000
order_id                 0.000000
order_dt                 0.000000
order_ts                 0.000000
currency_code            0.000000
revenue                  0.000000
tickets_count            0.000000
days_since_prev          7.547202
event_id                 0.000000
event_name               0.000000
event_type_main          0.000000
service_name             0.000000
region_name              0.000000
city_name                0.000000
curs                     0.000000
revenue_rub              0.000000
dtype: float64

In [37]:
#Проупски составляют 7% от данных, что достаточно много

In [39]:
#Разберемся в причине возникновения пропусков
df[df['days_since_prev'].isna()]

,user_id,device_type_canonical,order_id,order_dt,order_ts,currency_code,revenue,tickets_count,days_since_prev,event_id,event_name,event_type_main,service_name,region_name,city_name,curs,revenue_rub
0,0002849b70a3ce2,mobile,4359165,2024-08-20,2024-08-20 16:08:03,rub,1521.94,4,NaN,169230,f0f7b271-04eb-4af6-bcb8-8f05cf46d6ad,театр,Край билетов,Каменевский регион,Глиногорск,18.6972,1521.94
1,0005ca5e93f2cf4,mobile,7965605,2024-07-23,2024-07-23 18:36:24,rub,289.45,2,NaN,237325,40efeb04-81b7-4135-b41f-708ff00cc64c,выставки,Мой билет,Каменевский регион,Глиногорск,18.3419,289.45
3,000898990054619,mobile,1139875,2024-07-13,2024-07-13 19:40:48,rub,8.49,2,NaN,387271,2f638715-8844-466c-b43f-378a627c419f,другое,Лови билет!,Североярская область,Озёрск,18.5010,8.49
6,00096d1f542ab2b,desktop,6636941,2024-08-15,2024-08-15 16:48:48,rub,917.83,4,NaN,201953,2f98d69f-4e60-4ffc-8f16-e539383526b1,театр,Край билетов,Каменевский регион,Глиногорск,18.7730,917.83
7,000a55a418c128c,mobile,4657981,2024-09-29,2024-09-29 19:39:12,rub,47.78,1,NaN,265857,0d876e01-851e-458b-ba61-753e0e0c4063,театр,Лучшие билеты,Поленовский край,Дальнозолотск,19.3741,47.78
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
290565,fff13b79bd47d7c,mobile,2668813,2024-07-16,2024-07-16 22:17:10,rub,297.04,3,NaN,536131,3ccd92b9-5450-40a1-a7a8-5075cf90cc6a,другое,Мой билет,Каменевский регион,Глиногорск,18.4847,297.04
290574,fff32fc9ad0f9f6,desktop,2440931,2024-08-15,2024-08-15 14:36:28,rub,779.99,3,NaN,291439,9dbaefe5-56de-4b32-b991-7e9e4d04b702,стендап,Билеты без проблем,Каменевский регион,Глиногорск,18.7730,779.99
290576,fffcd3dde79eb2c,desktop,5114702,2024-06-20,2024-06-20 19:57:25,rub,367.56,1,NaN,529500,9303405c-c6df-4852-8768-73479f55dc25,концерты,Билеты без проблем,Каменевский регион,Глиногорск,18.0419,367.56
290609,fffeeb3c120cf0b,desktop,5526067,2024-09-24,2024-09-24 10:07:42,rub,661.53,2,NaN,454526,6f4c6a50-0106-407b-8f54-332c740b01da,стендап,Билеты без проблем,Широковская область,Ягодиновка,19.3967,661.53


При детальном рассмотрении стало ясно, что пропуски в этой колонке образуются, потому что пользователь не совершает повторных покупок. Эти данные являются важной частью датасета, поэтому их мы однозначно оставляем

**Работа с дубликатами**

In [42]:
#Считаем явные пропуски в наборе данных
df.duplicated().sum()

np.int64(0)

In [46]:
#Считаем дубликаты в колонке с заказами
df.duplicated(subset='order_id').sum()

np.int64(0)

In [51]:
#Создаем список из колонок с категориальными значениями
categorical_columns = ['device_type_canonical', 'currency_code', 'event_type_main', 'region_name', 'city_name']

In [52]:
#Выводим информацию о каждом категориальном поле с помощью цикла
for column in categorical_columns:
        print(f"\nColumn: {column}")
        print("Unique values:", df[column].unique())
        print("Missing values count:", df[column].isna().sum())
        print("Value counts:")
        print(df[column].value_counts(dropna=False), "\n") 


Column: device_type_canonical
Unique values: ['mobile' 'desktop']
Missing values count: 0
Value counts:
device_type_canonical
mobile     232490
desktop     58121
Name: count, dtype: int64 


Column: currency_code
Unique values: ['rub' 'kzt']
Missing values count: 0
Value counts:
currency_code
rub    285542
kzt      5069
Name: count, dtype: int64 


Column: event_type_main
Unique values: ['театр' 'выставки' 'другое' 'стендап' 'концерты' 'спорт' 'ёлки']
Missing values count: 0
Value counts:
event_type_main
концерты    115276
театр        67321
другое       65867
спорт        21911
стендап      13393
выставки      4854
ёлки          1989
Name: count, dtype: int64 


Column: region_name
Unique values: ['Каменевский регион' 'Североярская область' 'Озернинский край'
 'Лугоградская область' 'Поленовский край' 'Широковская область'
 'Медовская область' 'Златопольский округ' 'Малиновоярский округ'
 'Яблоневская область' 'Ветренский регион' 'Боровлянский край'
 'Крутоводская область' 'Ягодиновс

Категориальные поля в норме

**Уменьшение размерности для целочисленных показателей**

In [53]:
df.head(3)

,user_id,device_type_canonical,order_id,order_dt,order_ts,currency_code,revenue,tickets_count,days_since_prev,event_id,event_name,event_type_main,service_name,region_name,city_name,curs,revenue_rub
0,0002849b70a3ce2,mobile,4359165,2024-08-20,2024-08-20 16:08:03,rub,1521.94,4,NaN,169230,f0f7b271-04eb-4af6-bcb8-8f05cf46d6ad,театр,Край билетов,Каменевский регион,Глиногорск,18.6972,1521.94
1,0005ca5e93f2cf4,mobile,7965605,2024-07-23,2024-07-23 18:36:24,rub,289.45,2,NaN,237325,40efeb04-81b7-4135-b41f-708ff00cc64c,выставки,Мой билет,Каменевский регион,Глиногорск,18.3419,289.45
2,0005ca5e93f2cf4,mobile,7292370,2024-10-06,2024-10-06 13:56:02,rub,1258.57,4,75.0,578454,01f3fb7b-ed07-4f94-b1d3-9a2e1ee5a8ca,другое,За билетом!,Каменевский регион,Глиногорск,19.6475,1258.57


In [ ]:
#Создаем список из колонок с целочисленными показателями
numerical_columns = ['order_id', 'tickets_count', 'event_id']

In [57]:
#Уменьшаем размерность для всех колонок в списке с помощью цикла
for column in numerical_columns:
    df[column] = pd.to_numeric(df[column], downcast='integer')

In [58]:
#Проверяем результат уменьшения размерности
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 290611 entries, 0 to 290610
Data columns (total 17 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   user_id                290611 non-null  object        
 1   device_type_canonical  290611 non-null  object        
 2   order_id               290611 non-null  int32         
 3   order_dt               290611 non-null  datetime64[ns]
 4   order_ts               290611 non-null  datetime64[ns]
 5   currency_code          290611 non-null  object        
 6   revenue                290611 non-null  float64       
 7   tickets_count          290611 non-null  int8          
 8   days_since_prev        268678 non-null  float64       
 9   event_id               290611 non-null  int32         
 10  event_name             290611 non-null  object        
 11  event_type_main        290611 non-null  object        
 12  service_name           290611 non-null  obje